# Component Construction

This page shows the most direct path for building a custom source when you already have a master-equation model in mind. The new `Emitter.from_master_equation(...)` and `Source.from_master_equation(...)` entry points are designed for users coming from QuTiP: you provide a Hamiltonian, a set of monitored collapse operators, any additional environment operators, and an initial state. ZPG then turns that model into an emitter or source with the usual source-quality and processor interfaces.

For source construction, monitored channels are special in two ways: they contribute to the dissipative master equation, and they define the exposed source output ports. When a source has more than one monitored channel, it is often best to pass them as a dictionary so the output ports get stable names immediately.

## Imports

We will use the top-level ZPG imports together with a few basic QuTiP operators.

In [1]:
from zpgenerator import *
from qutip import basis, destroy, create
import numpy as np

## Example 1. A custom driven two-level source

Suppose we already know the master equation of a driven two-level system. The monitored collapse operator is the radiative decay channel that we want to collect as source output. Since the Gaussian pulse has finite time support, `Source.from_master_equation(...)` can infer a reasonable default source gate if we opt in explicitly with `infer_gate=True`.

In [2]:
states = {'|g>': basis(2, 0), '|e>': basis(2, 1)}
operators = {
    'lower': destroy(2),
    'raise': create(2),
    'number': create(2) * destroy(2),
}

pulse = Pulse.gaussian(parameters={'width': 0.2, 'area': np.pi})
drive = TimeOperator(operator=(operators['lower'] + operators['raise']) / 2, functions=pulse)

source = Source.from_master_equation(
    hamiltonian=drive,
    monitored=[operators['lower']],
    initial_state='|g>',
    infer_gate=True,
    states=states,
    operators=operators,
    name='custom source',
)

source.default_parameters

{'width': 0.2,
 'area': 3.141592653589793,
 'delay': 0,
 'detuning': 0,
 'phase': 0,
 'window': 6,
 'efficiency': 1}

In [3]:
source.times(), source.output.open_modes

([-1.2000000000000002, 1.2000000000000002], 1)

In [4]:
source.beta(), source.mu()

(0.6860592714826821, 0.7233010102061677)

The resulting object behaves like any other catalogue source. It can be passed into a `Processor`, queried with the standard source-quality methods, or composed into larger photonic networks.

## Example 2. A multi-port source from monitored channels

For a user coming from QuTiP, the most important extra concept in ZPG is the distinction between ordinary environment operators and *monitored* operators. The monitored operators both contribute to the dissipative master equation and define source output ports.

In the example below, we build a simple three-level V-system with two monitored decay channels. Because the model is time-independent, we provide an explicit gate. By passing the monitored operators as a dictionary, the two source outputs are named `x` and `y`.

In [5]:
states_v = {'|g>': basis(3, 0), '|x>': basis(3, 1), '|y>': basis(3, 2)}
lower_x = states_v['|g>'] * states_v['|x>'].dag()
lower_y = states_v['|g>'] * states_v['|y>'].dag()
initial = (states_v['|x>'] + states_v['|y>']).unit()

multiport = Source.from_master_equation(
    hamiltonian=0 * (lower_x.dag() * lower_x),
    monitored={'x': lower_x, 'y': lower_y},
    initial_state=initial,
    states=states_v,
    operators={'lower_x': lower_x, 'lower_y': lower_y},
    gate=[0, 10],
    name='v source',
)

multiport.output.port_names, multiport.output.open_port_names

(['x', 'y'], ['x', 'y'])

In [6]:
multiport.beta('x'), multiport.beta('y')

(0.49997730003511875, 0.49997730003511875)

## When to use each construction path

- Use `Source.from_master_equation(...)` when you already have a QuTiP-style model and want a source quickly.
- Use `Emitter.from_master_equation(...)` when you want to build an emitter first and then wrap or compose it yourself.
- Use the catalogue factories when one of the built-in physical models already matches what you need.

This keeps the user-facing workflow simple while still leaving the lower-level emitter, control, and component abstractions available for more specialized models.